# E-commerce Delta ETL
Generates synthetic source data, runs the batch pipeline, and displays the output tables.

In [0]:
from pathlib import Path
import sys
import uuid

project_root = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "pyproject.toml").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Could not locate pyproject.toml. Open this notebook from the project Git folder."
    )
sys.path.insert(0, str(project_root))

from ecommerce_etl.generate_data import generate_raw_data
from ecommerce_etl.pipeline import run_pipeline

In [0]:
dbutils.widgets.text("raw_path", "/Volumes/workspace/default/ecommerce_etl_raw/validation-run-2026-11-24")
dbutils.widgets.text("database", "ecommerce_dw_validation")
dbutils.widgets.text("order_count", "50000")

raw_path = dbutils.widgets.get("raw_path").rstrip("/")
database = dbutils.widgets.get("database")
order_count = int(dbutils.widgets.get("order_count"))
batch_id = str(uuid.uuid4())

generated_counts = generate_raw_data(spark, raw_path, order_count=order_count)
print(f"Generated source records: {generated_counts}")
print(f"Processing batch: {batch_id}")

Generated source records: {'customers': 7500, 'products': 350, 'orders': 52492}
Processing batch: 4a0e6026-7cc4-4e28-9a8e-934e148b95a8


In [0]:
dq_results = run_pipeline(spark, raw_path, database, batch_id)
print(f"Data-quality checks passed: {dq_results}")

Data-quality checks passed: {'fact_rows': 45608, 'null_order_ids': 0, 'duplicate_order_ids': 0, 'invalid_amounts': 0, 'missing_customers': 0, 'missing_products': 0}


In [0]:
display(spark.table(f"{database}.agg_daily_category_kpis").orderBy("order_date", "category"))

order_date,category,total_orders,unique_customers,net_revenue,returned_orders,return_rate_pct
2025-01-01,Books,24,24,168677.97999999998,7,29.17
2025-01-01,Electronics,30,30,190531.80000000002,5,16.67
2025-01-01,Fashion,21,21,106002.63,6,28.57
2025-01-01,Fitness,17,17,122354.54999999999,2,11.76
2025-01-01,Home & Kitchen,29,29,299207.10000000003,3,10.34
2025-01-02,Books,19,19,172419.85,2,10.53
2025-01-02,Electronics,26,26,162220.67000000004,4,15.38
2025-01-02,Fashion,28,28,210680.61999999997,5,17.86
2025-01-02,Fitness,29,29,138678.80000000002,8,27.59
2025-01-02,Home & Kitchen,27,27,286702.15,3,11.11


In [0]:
display(spark.table(f"{database}.quarantine_orders").filter(f"batch_id = '{batch_id}'").limit(50))

order_id,customer_id,product_id,quantity,unit_price_raw,order_timestamp_raw,city,order_status,batch_id,ingested_at,source_file,unit_price_clean,unit_price,quantity_int,order_ts,quarantine_reason,quarantined_at
ORD-100192,null,PROD-254,1,14208.69,2025-06-15 06:51:00,Chennai,COMPLETED,4a0e6026-7cc4-4e28-9a8e-934e148b95a8,2026-11-24T06:40:31.741Z,dbfs:/Volumes/workspace/default/ecommerce_etl_raw/validation-run-2026-11-24/orders/part-00000-tid-6690103271938086536-fdd7c0ea-b072-467c-a5d1-558763abafac-1075-1-c000.csv,14208.69,14208.69,1,2025-06-15T06:51:00.000Z,missing customer_id,2026-11-24T06:40:41.283Z
ORD-100504,CUST-1347,PROD-231,-1,13848.65,2025-12-28 20:03:00,Pune,RETURNED,4a0e6026-7cc4-4e28-9a8e-934e148b95a8,2026-11-24T06:40:31.741Z,dbfs:/Volumes/workspace/default/ecommerce_etl_raw/validation-run-2026-11-24/orders/part-00000-tid-6690103271938086536-fdd7c0ea-b072-467c-a5d1-558763abafac-1075-1-c000.csv,13848.65,13848.65,-1,2025-12-28T20:03:00.000Z,invalid quantity,2026-11-24T06:40:41.283Z
ORD-100519,null,PROD-388,2,"₹5,839.13",2025-09-16 22:03:00,Chennai,CANCELLED,4a0e6026-7cc4-4e28-9a8e-934e148b95a8,2026-11-24T06:40:31.741Z,dbfs:/Volumes/workspace/default/ecommerce_etl_raw/validation-run-2026-11-24/orders/part-00000-tid-6690103271938086536-fdd7c0ea-b072-467c-a5d1-558763abafac-1075-1-c000.csv,5839.13,5839.13,2,2025-09-16T22:03:00.000Z,missing customer_id,2026-11-24T06:40:41.283Z
ORD-100678,CUST-2704,PROD-436,0,9535.31,2025-10-30 16:37:00,UNKNOWN,PENDING,4a0e6026-7cc4-4e28-9a8e-934e148b95a8,2026-11-24T06:40:31.741Z,dbfs:/Volumes/workspace/default/ecommerce_etl_raw/validation-run-2026-11-24/orders/part-00000-tid-6690103271938086536-fdd7c0ea-b072-467c-a5d1-558763abafac-1075-1-c000.csv,9535.31,9535.31,0,2025-10-30T16:37:00.000Z,invalid quantity,2026-11-24T06:40:41.283Z
ORD-100740,null,PROD-142,1,12122.13,2025-05-02 00:58:30,Pune,COMPLETED,4a0e6026-7cc4-4e28-9a8e-934e148b95a8,2026-11-24T06:40:31.741Z,dbfs:/Volumes/workspace/default/ecommerce_etl_raw/validation-run-2026-11-24/orders/part-00000-tid-6690103271938086536-fdd7c0ea-b072-467c-a5d1-558763abafac-1075-1-c000.csv,12122.13,12122.13,1,2025-05-02T00:58:30.000Z,missing customer_id,2026-11-24T06:40:41.283Z
ORD-100801,CUST-8198,PROD-207,0,"₹3,935.93",2025-10-02 10:51:00,Chennai,CANCELLED,4a0e6026-7cc4-4e28-9a8e-934e148b95a8,2026-11-24T06:40:31.741Z,dbfs:/Volumes/workspace/default/ecommerce_etl_raw/validation-run-2026-11-24/orders/part-00000-tid-6690103271938086536-fdd7c0ea-b072-467c-a5d1-558763abafac-1075-1-c000.csv,3935.93,3935.93,0,2025-10-02T10:51:00.000Z,invalid quantity,2026-11-24T06:40:41.283Z
ORD-100825,CUST-2014,PROD-152,3,12339.01,not-a-timestamp,Chennai,CANCELLED,4a0e6026-7cc4-4e28-9a8e-934e148b95a8,2026-11-24T06:40:31.741Z,dbfs:/Volumes/workspace/default/ecommerce_etl_raw/validation-run-2026-11-24/orders/part-00000-tid-6690103271938086536-fdd7c0ea-b072-467c-a5d1-558763abafac-1075-1-c000.csv,12339.01,12339.01,3,null,invalid order timestamp,2026-11-24T06:40:41.283Z
ORD-101249,null,PROD-229,3,9101.05,2025-12-18 17:40:00,Pune,RETURNED,4a0e6026-7cc4-4e28-9a8e-934e148b95a8,2026-11-24T06:40:31.741Z,dbfs:/Volumes/workspace/default/ecommerce_etl_raw/validation-run-2026-11-24/orders/part-00000-tid-6690103271938086536-fdd7c0ea-b072-467c-a5d1-558763abafac-1075-1-c000.csv,9101.05,9101.05,3,2025-12-18T17:40:00.000Z,missing customer_id,2026-11-24T06:40:41.283Z
ORD-101445,null,PROD-187,1,8914.01,2025-12-05 13:59:00,Mumbai,COMPLETED,4a0e6026-7cc4-4e28-9a8e-934e148b95a8,2026-11-24T06:40:31.741Z,dbfs:/Volumes/workspace/default/ecommerce_etl_raw/validation-run-2026-11-24/orders/part-00000-tid-6690103271938086536-fdd7c0ea-b072-467c-a5d1-558763abafac-1075-1-c000.csv,8914.01,8914.01,1,2025-12-05T13:59:00.000Z,missing customer_id,2026-11-24T06:40:41.283Z
ORD-101473,null,PROD-130,2,12481.44,2025-01-27 13:29:00,Pune,COMPLETED,4a0e6026-7cc4-4e28-9a8e-934e148b95a8,2026-11-24T06:40:31.741Z,dbfs:/Volumes/workspace/default/ecommerce_etl_raw/validation-run-2026-11-24/orders/part-00000-tid-66901032719380